In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sentence_transformers import SentenceTransformer
import citation_utils
import metric_utils
import reranker_utils
import rrf


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS
/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefin

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

# model = SentenceTransformer("/root/.cache/modelscope/hub/models/Qwen/Qwen3-Embedding-0___6B", model_kwargs={"torch_dtype": "float16"})

# dense_index = DenseIndex(model, "../data/processed/_dense_court", court_doc)
dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
from tqdm import tqdm

hits_l = []
gold_citations_l = []

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

for query, gold_citations in tqdm(zip(valid_df['query2'].tolist(), valid_df['gold_citations']), total=len(valid_df)):
    hits = dense_index.search(query, top_k=1000)
    # hits_l.append(hits)
    gold_citations_l.append(gold_citations.split(";"))

    hit_with_score_l = reranker_utils.rerank_by_dense_batch_chunked(reranker, query, hits, 1000, 10, 384, 128)

    rrf_score_l = rrf.compute2_with_score([[hit['citation'] for hit,_ in hit_with_score_l]], k=60, top_k=1000)

    print(rrf_score_l[0])
    first_layer = []
    for citation,score in rrf_score_l:
        if citation in court_consideration_d:
            first_layer.append(({'citation':citation, 'text':court_consideration_d[citation]}, score))
        # elif citation in law_d:
        #     first_layer.append(({'citation':citation, 'text':law_d[citation]}, score))
        
    second_layer = citation_utils.compute_citation_score_with_sentence_pos(first_layer)

    hits = []
    for citation,score in second_layer:
        #if citation in court_consideration_d:
        #    hits.append({'citation':citation, 'text':court_consideration_d[citation]})
        if citation in law_d:
            hits.append({'citation':citation, 'text':law_d[citation]})

    print("second_layer.len:", len(second_layer), 'first_layer.len:', len(first_layer), 'hits.len:', len(hits))

    
    hits_l.append(hits)

  0%|          | 0/10 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
 10%|█         | 1/10 [00:25<03:49, 25.52s/it]

('7B_1440/2024 05.02.2025 E. B', 0.01639344262295082)
second_layer.len: 662 first_layer.len: 993 hits.len: 190
('U 333/02 11.08.2003 E. 3', 0.01639344262295082)


 20%|██        | 2/10 [00:58<04:00, 30.11s/it]

second_layer.len: 765 first_layer.len: 987 hits.len: 95


 30%|███       | 3/10 [01:33<03:46, 32.36s/it]

...

second_layer.len: 2755 first_layer.len: 981 hits.len: 349
('BGE 145 V 154 E. 4.1.2', 0.01639344262295082)


 90%|█████████ | 9/10 [05:47<00:41, 41.69s/it]

second_layer.len: 1498 first_layer.len: 984 hits.len: 296
('6S.179/2006 28.06.2006 E. 3', 0.01639344262295082)


100%|██████████| 10/10 [06:42<00:00, 40.25s/it]

second_layer.len: 2255 first_layer.len: 948 hits.len: 331


In [5]:
import math
for limit in [5,15,20,25,30,35,40,45]:
    r = metric_utils.cal_recall(hits_l, gold_citations_l, lambda x: max(limit, int(math.log(len(x)))))
    p = metric_utils.cal_precision(hits_l, gold_citations_l, lambda x: max(limit, int(math.log(len(x)))))
    f1 = metric_utils.cal_f1(r,p)
    print(f"[{limit}] r:",r, "p:",p, "f1:",f1)

[5] r: 0.07960152972528925 p: 0.31333333333333335 f1: 0.126951385545732
[15] r: 0.15455173416767423 p: 0.2066666666666667 f1: 0.17684974881794238
[20] r: 0.16169459131053138 p: 0.16 f1: 0.16084283235406746
[25] r: 0.16847236908830915 p: 0.13599999999999998 f1: 0.1505045746161917
[30] r: 0.18077395638989646 p: 0.12333333333333336 f1: 0.1466288732618795
[35] r: 0.19159266984018886 p: 0.11428571428571428 f1: 0.14317000651853998
[40] r: 0.20030410359699466 p: 0.10500000000000001 f1: 0.13777692883844667
[45] r: 0.21615839449665764 p: 0.09999999999999999 f1: 0.13674056944829457
